In [ ]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# Resolve paths from the repository root or this notebook's folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'playground-series-s6e9' / 'train.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Download the competition CSVs into playground-series-s6e9/.')
DATA_DIR = PROJECT_ROOT / 'playground-series-s6e9'
OUTPUT_DIR = PROJECT_ROOT / 'CatBoost'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Preprocessing
target = 'Will_Buy_EV'
features = [c for c in train.columns if c not in ['id', target]]

cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train[target] = train[target].map({'Yes': 1, 'No': 0})

X = train[features]
y = train[target]
X_test = test[features]

num_trials = 50
n_splits = 5


In [ ]:
from catboost import CatBoostClassifier, Pool

def objective(trial):
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'auto_class_weights': 'Balanced',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'verbose': False,
        'task_type': 'CPU'
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        train_pool = Pool(X_tr, y_tr)
        val_pool = Pool(X_val, y_val)
        
        model = CatBoostClassifier(**params, iterations=1000, early_stopping_rounds=50)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
        preds = model.predict_proba(val_pool)[:, 1]
        auc = roc_auc_score(y_val, preds)
        auc_scores.append(auc)
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=num_trials)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params['loss_function'] = 'Logloss'
best_params['eval_metric'] = 'AUC'
best_params['auto_class_weights'] = 'Balanced'
best_params['verbose'] = False

with open(OUTPUT_DIR / 'catboost_best_params.json', 'w') as f:
    json.dump(best_params, f)


In [ ]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_preds_df = pd.DataFrame({'id': test['id']})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr)
    val_pool = Pool(X_val, y_val)
    test_pool = Pool(X_test)
    
    model = CatBoostClassifier(**best_params, iterations=2000, early_stopping_rounds=50)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
    
    val_preds = model.predict_proba(val_pool)[:, 1]
    oof_preds[val_idx] = val_preds
    fold_test_preds = model.predict_proba(test_pool)[:, 1]
    test_preds += fold_test_preds / n_splits
    fold_preds_df[f'fold_{fold}'] = fold_test_preds

print("OOF AUC:", roc_auc_score(y, oof_preds))

pd.DataFrame({'id': train['id'], 'Will_Buy_EV': oof_preds}).to_csv(OUTPUT_DIR / 'catboost_oof_predictions.csv', index=False)
fold_preds_df.to_csv(OUTPUT_DIR / 'catboost_fold_predictions.csv', index=False)

sample_sub['Will_Buy_EV'] = test_preds
sample_sub.to_csv(OUTPUT_DIR / 'submission_catboost.csv', index=False)
